In [33]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
import traceback

from openpyxl import Workbook
from openpyxl.styles import (Font, PatternFill, Alignment,
                              Border, Side)
from openpyxl.utils import get_column_letter

In [ ]:
# 0. LOAD DATA FROM variable selection
print("Loading data from Part 1...")
df = pd.read_csv("outputs/recoded_analysis_data.csv")
print(f"Shape: {df.shape}")

df.shape

df["age"].describe()

Loading data from Part 1...
Shape: (1533, 25)


count    1533.000000
mean       30.849315
std         7.602485
min        16.000000
25%        25.000000
50%        30.000000
75%        36.000000
max        49.000000
Name: age, dtype: float64

In [35]:
# ── OUTCOME ───────────────────────────────────────────────
OUTCOME = 'fp_decision_autonomy'

# Keep only valid outcome codes
df = df[df[OUTCOME].isin([1, 2, 3, 6])].copy()
df[OUTCOME] = df[OUTCOME].astype(int)

# Binary outcome (for Lasso — Demissie approach)
df['autonomy_binary'] = df[OUTCOME].map({1:1, 3:1, 2:0, 6:0})

OUTCOME_LABELS = {
    1: 'Woman alone',
    2: 'Partner alone',
    3: 'Joint decision',
    6: 'Other'
}

print(f"\nAnalytical sample: {len(df)}")
print("Outcome distribution:")
print(df[OUTCOME].value_counts().sort_index())

# ── VARIABLE DEFINITIONS ──────────────────────────────────
# Format: (column_name, display_label, level)
# Level drives which model the variable enters:
#   'Individual'      → Model I
#   'Community'       → Model II
#   Both              → Model III

VAR_DEFS = [
    # ── INDIVIDUAL LEVEL ──────────────────────────────
    ('age_group',             'Age group',                       'Individual'),
    ('marital_status',        'Marital status',                  'Individual'),
    ('edu_woman',             "Woman's education",               'Individual'),
    ('wealth',                'Wealth index',                    'Individual'),
    # ('marriage_type',         'Type of marriage (polygyny)',     'Individual'),
    ('woman_working',         'Woman currently working',         'Individual'),
    ('fertility_preference',  'Fertility preference',            'Individual'),
    ('children_group',        'Number of living children',       'Individual'),
    ('current_method',        'Current contraceptive method',    'Individual'),
    ('edu_husband',           "Partner's education",             'Individual'),
    ('husband_working',       'Partner currently working',       'Individual'),
    ('husband_desired_children', "Partner's desired children",   'Individual'),
    ('anc_group',             'ANC visits (last pregnancy)',     'Individual'),
   ('facility_fp',           'Facility staff talked about FP',  'Individual'),
    ('media_any',             'Exposed to FP media message',     'Individual'),

    # ── COMMUNITY LEVEL ───────────────────────────────
    ('residence',             'Residence (urban/rural)',         'Community'),
    ('region',                'Region',                          'Community'),
    ('religion',              'Religion',                        'Community'),
    ('fieldworker_fp',        'Fieldworker talked about FP',     'Community'),
    
]

var_labels = {v: l for v, l, _ in VAR_DEFS}
INDIVIDUAL_VARS = [v for v, l, lvl in VAR_DEFS if lvl == 'Individual']
COMMUNITY_VARS  = [v for v, l, lvl in VAR_DEFS if lvl == 'Community']

print(f"\nIndividual-level vars: {len(INDIVIDUAL_VARS)}")
print(f"Community-level vars:  {len(COMMUNITY_VARS)}")


Analytical sample: 1533
Outcome distribution:
fp_decision_autonomy
1    416
2    238
3    877
6      2
Name: count, dtype: int64

Individual-level vars: 14
Community-level vars:  4


In [36]:


print("STEP 1 — DESCRIPTIVE STATISTICS")


def weighted_freq(series, weights, label_map=None):
    """Weighted frequency table with optional label mapping."""
    tmp = pd.DataFrame({'val': series, 'w': weights}).dropna()
    result = tmp.groupby('val')['w'].sum()
    pct = result / result.sum() * 100
    out = pd.DataFrame({
        'N (weighted)': result.round(0).astype(int),
        'Percent (%)':  pct.round(1)
    })
    if label_map:
        out.index = out.index.map(lambda x: label_map.get(x, x))
    return out

# Value label maps for readable output
LABEL_MAPS = {
    'age_group':            {1:'15–19', 2:'20–24', 3:'25–29', 4:'30–34',
                             5:'35–39', 6:'40–44', 7:'45–49'},
    'residence':            {1:'Urban', 2:'Rural'},
    'region':               {1: 'Adamawa',2: 'Centre (without Yaounde)',3: 'Douala',
                                4: 'East',5: 'Far-North',6: 'Littoral (without Douala)',7: 'North',
                                8: 'North-West',9: 'West',10: 'South',
                                11: 'South-West',12: 'Yaounde'
                            },
    'edu_woman':            {0:'No education', 1:'Primary',
                             2:'Secondary',    3:'Higher'},
    'edu_husband':          {0:'No education', 1:'Primary',
                             2:'Secondary',    3:'Higher' ,8:'Dont Know'},
    'wealth':               {1:'Poorest', 2:'Poorer', 3:'Middle',
                             4:'Richer',  5:'Richest'},
    'religion':             {1: 'Catholic',
                                2: 'Protestant',
                                3: 'Other Christians',
                                4: 'Muslim',
                                5: 'Animist',
                                7: 'None',
                                96: 'Other'},
    'marital_status':       {1:'Formally married', 2:'Cohabiting'},
    # 'marriage_type':        {1:'Monogamous', 2:'Polygynous',
    #                          3:'Not Married'},
    'woman_working':        {0:'Not working', 1:'Working'},
    'husband_working':      {0:'Not working', 1:'Working' ,2:'Dont Know'},
    'husband_desired_children': {3:'Wants fewer', 1:'Wants same',
                                2:'Wants more', 8:'Dont Know'},
    'children_group':       {1:' <3 children', 2:'3–5 children',
                             3:'>5 children'},

    'fertility_preference': {1:'Wants More', 2:'Doesn\'t want more/ Undecided',
                             3:'Cannot have children', 4:'Never had sex'},
    'anc_group':            {1:'No ANC', 2:'1–3 visits',
                             3:'>=4 visits', 0:'No pregnancy in last 5 years'},

    'facility_fp':          {0:'No', 1:'Yes',2:'Didn\'t visit facility'},
    'media_any':            {0:'No', 1:'Yes'},
    'current_method':         {0.0: 'Not using',
                                1.0: 'Pill',
                                2.0: 'IUD',
                                3.0: 'Injections',
                                4.0: 'Diaphragm',
                                5.0: 'Male condom',
                                6.0: 'Female sterilization',
                                7.0: 'Male sterilization',
                                8.0: 'Periodic abstinence',
                                9.0: 'Withdrawal',
                                10.0: 'Other traditional',
                                11.0: 'Implants/Norplant',
                                12.0: 'Prolonged abstinence',
                                13.0: 'Lactational amenorrhea (LAM)',
                                14.0: 'Female condom',
                                15.0: 'Foam or jelly',
                                16.0: 'Emergency contraception',
                                17.0: 'Other modern method',
                                18.0: 'Standard days method (SDM)',
                                19.0: 'Specific method 1',
                                20.0: 'Specific method 2'},
    'fieldworker_fp':       {0:'No', 1:'Yes',2:'Didn\'t visit facility'},
}

desc_tables = {}

# Outcome
desc_tables['_outcome'] = weighted_freq(
    df[OUTCOME].map(OUTCOME_LABELS), df['weight'])
print("\n📊 Outcome distribution:")
print(desc_tables['_outcome'])

# Each variable
for var, label, _ in VAR_DEFS:
    if var not in df.columns:
        continue
    lmap = LABEL_MAPS.get(var)
    tab  = weighted_freq(df[var], df['weight'], lmap)
    desc_tables[var] = tab

STEP 1 — DESCRIPTIVE STATISTICS

📊 Outcome distribution:
                N (weighted)  Percent (%)
val                                      
Joint decision           858         57.3
Other                      1          0.1
Partner alone            248         16.6
Woman alone              389         26.0


In [37]:

print("STEP 2 — BIVARIATE ANALYSIS")


def cramers_v(x, y):
    ct   = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n    = ct.sum().sum()
    r, k = ct.shape
    if min(k-1, r-1) == 0:
        return np.nan
    return np.sqrt(chi2 / n / min(k-1, r-1))

bivariate_rows = []

for var, label, level in VAR_DEFS:
    if var not in df.columns:
        continue
    tmp = df[[var, OUTCOME]].dropna()
    if len(tmp) < 20:
        continue
    try:
        ct = pd.crosstab(tmp[var], tmp[OUTCOME])
        chi2_val, p, dof, _ = chi2_contingency(ct)
        cv  = cramers_v(tmp[var], tmp[OUTCOME])
        sig = ('***' if p < 0.001 else
               ('**' if p < 0.01 else
               ('*'  if p < 0.05 else
               ('†'  if p < 0.20 else 'ns'))))
        bivariate_rows.append({
            'Level':          level,
            'Variable':       label,
            'Chi-square':     round(chi2_val, 3),
            'df':             int(dof),
            'p-value':        round(p, 4),
            'Sig.':           sig,
            "Cramér's V":     round(cv, 4),
            'Enter (p<0.20)': 'YES' if p < 0.20 else 'NO'
        })
        print(f"  {label:42s}  χ²={chi2_val:9.3f}  p={p:.4f}  {sig:3s}  V={cv:.3f} Enter Model={p<0.20}")
    except Exception as e:
        print(f"  {label}: ERROR — {e}")

bivariate_df = pd.DataFrame(bivariate_rows)
passed_labels = bivariate_df[
    bivariate_df['Enter (p<0.20)'] == 'YES']['Variable'].tolist()
passed_vars = [v for v, l, _ in VAR_DEFS
               if l in passed_labels and v in df.columns]
print(f"\n✅ Variables passing p<0.20: {len(passed_vars)}")

STEP 2 — BIVARIATE ANALYSIS
  Age group                                   χ²=   23.394  p=0.1759  †    V=0.071 Enter Model=True
  Marital status                              χ²=   25.176  p=0.0000  ***  V=0.128 Enter Model=True
  Woman's education                           χ²=   36.160  p=0.0000  ***  V=0.089 Enter Model=True
  Wealth index                                χ²=   32.609  p=0.0011  **   V=0.084 Enter Model=True
  Woman currently working                     χ²=    8.257  p=0.0410  *    V=0.073 Enter Model=True
  Fertility preference                        χ²=   23.764  p=0.0006  ***  V=0.088 Enter Model=True
  Number of living children                   χ²=   13.943  p=0.0303  *    V=0.067 Enter Model=True
  Current contraceptive method                χ²=    1.448  p=0.6943  ns   V=0.031 Enter Model=False
  Partner's education                         χ²=   29.814  p=0.0030  **   V=0.081 Enter Model=True
  Partner currently working                   χ²=    5.146  p=0.5252  n

In [38]:

def build_X(df_in, var_list):
    """
    Build dummy-encoded X matrix from variable list.
    Ordinal vars with >2 unique values → dummies (drop_first).
    Binary vars → kept as numeric.
    Returns X_matrix (with const) and column names.
    """
    parts = []
    for v in var_list:
        col = df_in[v]
        nu  = col.nunique()
        if col.dtype == 'object' or str(col.dtype) == 'category':
            d = pd.get_dummies(col, prefix=v, drop_first=True)
            parts.append(d.astype(float))
        elif nu > 2:
            d = pd.get_dummies(col, prefix=v, drop_first=True)
            parts.append(d.astype(float))
        else:
            parts.append(col.rename(v).astype(float))
    if not parts:
        return None, []
    X = pd.concat(parts, axis=1)
    X = sm.add_constant(X)
    return X, X.columns.tolist()


def fit_mnlogit(df_model, var_list):
    """
    Fit MNLogit with reference = 1 (Woman alone → encoded as 0).
    Returns fitted model or None on failure.
    """
    outcome_encode = {1:0, 2:1, 3:2, 6:3}
    dm = df_model[var_list + [OUTCOME]].dropna().copy()
    dm['y'] = dm[OUTCOME].map(outcome_encode).astype(int)
    X, _ = build_X(dm, var_list)
    if X is None:
        return None, None
    y = dm['y']
    try:
        model = sm.MNLogit(y, X).fit(
            method='bfgs', maxiter=2000, disp=False)
        return model, dm
    except Exception as e:
        print(f"  Model fit error: {e}")
        return None, None


def model_metrics(model, dm):
    """Extract key goodness-of-fit metrics."""
    n = len(dm)
    k = model.df_model + 1
    llf   = model.llf
    llnull= model.llnull
    aic   = model.aic
    bic   = model.bic
    pr2   = 1 - llf / llnull          # McFadden pseudo-R²
    cox   = 1 - np.exp(-2*(llf-llnull)/n)  # Cox & Snell
    nag   = cox / (1 - np.exp(2*llnull/n)) # Nagelkerke
    dev   = -2 * llf                  # Deviance
    lrt   = -2 * (llnull - llf)       # LR statistic
    return {
        'N': n,
        'Log-likelihood': round(llf, 3),
        'Null log-likelihood': round(llnull, 3),
        'Deviance': round(dev, 3),
        'AIC': round(aic, 3),
        'BIC': round(bic, 3),
        'McFadden R²': round(pr2, 4),
        'Cox & Snell R²': round(cox, 4),
        'Nagelkerke R²': round(nag, 4),
        'LR statistic': round(lrt, 3),
    }

In [39]:
final_comm = [v for v in COMMUNITY_VARS if v != "region"]

final_indiv = [
    v for v in INDIVIDUAL_VARS 
    if v not in ["current_method", "husband_working"]
]

final_comm

['residence', 'religion', 'fieldworker_fp']

In [40]:

print("\n" + "="*60)
print("STEP 3 — FOUR MULTINOMIAL MODELS")
print("Reference: Woman alone (1)")
print("="*60)



# ── Null model ────────────────────────────────────────────
print("\n[Null model] — intercept only")
outcome_encode = {1:0, 2:1, 3:2, 6:3}
dm_null = df[[OUTCOME]].dropna().copy()
dm_null['y'] = dm_null[OUTCOME].map(outcome_encode).astype(int)
X_null  = sm.add_constant(
    pd.DataFrame({'const_only': np.ones(len(dm_null))},
                 index=dm_null.index))
X_null  = X_null.drop(columns=['const_only'])  # intercept only via add_constant
X_null  = sm.add_constant(
    pd.DataFrame(index=dm_null.index))

# statsmodels null: just const
X_null = pd.DataFrame({'const': 1.0},
                      index=dm_null.index)
null_model = sm.MNLogit(dm_null['y'], X_null).fit(
    method='bfgs', maxiter=1000, disp=False)
print(f"  Deviance: {-2*null_model.llf:.3f}")

# ── Model I: Individual-level only ───────────────────────
print("\n[Model I] — Individual-level variables")
model_I, dm_I = fit_mnlogit(df, final_indiv)
if model_I:
    print(f"  Deviance: {-2*model_I.llf:.3f}  AIC: {model_I.aic:.3f}")

# ── Model II: Community-level only ───────────────────────
print("\n[Model II] — Community-level variables")
model_II, dm_II = fit_mnlogit(df, final_comm)
if model_II:
    print(f"  Deviance: {-2*model_II.llf:.3f}  AIC: {model_II.aic:.3f}")

# ── Model III: Both levels (final model) ─────────────────
print("\n[Model III] — Individual + Community (final model)")
model_III, dm_III = fit_mnlogit(df, passed_vars)
if model_III:
    print(f"  Deviance: {-2*model_III.llf:.3f}  AIC: {model_III.aic:.3f}")
    print(model_III.summary())

# ── Model comparison table ────────────────────────────────
metrics_null = {
    'N':                   len(dm_null),
    'Log-likelihood':      round(null_model.llf, 3),
    'Null log-likelihood': round(null_model.llnull, 3),
    'Deviance':            round(-2*null_model.llf, 3),
    'AIC':                 round(null_model.aic, 3),
    'BIC':                 round(null_model.bic, 3),
    'McFadden R²':         '—',
    'Cox & Snell R²':      '—',
    'Nagelkerke R²':       '—',
    'LR statistic':        '—',
}

comparison_data = {
    'Null model':                        metrics_null,
    'Model I (Individual)':              model_metrics(model_I,   dm_I)   if model_I   else {},
    'Model II (Community)':              model_metrics(model_II,  dm_II)  if model_II  else {},
    'Model III (Individual+Community)':  model_metrics(model_III, dm_III) if model_III else {},
}

comparison_df = pd.DataFrame(comparison_data).T
print("\n📊 Model Comparison:")
print(comparison_df)

# ── Identify best model ───────────────────────────────────
# Best = lowest AIC (among non-null models)
aics = {}
for name, m in [('Model I', model_I), ('Model II', model_II),
                ('Model III', model_III)]:
    if m:
        aics[name] = m.aic
best_model_name = min(aics, key=aics.get)
print(f"\n✅ Best model by AIC: {best_model_name} (AIC={aics[best_model_name]:.3f})")
# Model III is expected to be best — use it as final
final_model = model_III
dm_final    = dm_III


STEP 3 — FOUR MULTINOMIAL MODELS
Reference: Woman alone (1)

[Null model] — intercept only
  Deviance: 2977.958

[Model I] — Individual-level variables
  Deviance: 2672.007  AIC: 2870.007

[Model II] — Community-level variables
  Deviance: 2932.373  AIC: 2992.373

[Model III] — Individual + Community (final model)
  Deviance: 2554.555  AIC: 2836.555
                          MNLogit Regression Results                          
Dep. Variable:                      y   No. Observations:                 1510
Model:                        MNLogit   Df Residuals:                     1369
Method:                           MLE   Df Model:                          138
Date:                Thu, 16 Apr 2026   Pseudo R-squ.:                  0.1311
Time:                        10:15:42   Log-Likelihood:                -1277.3
converged:                       True   LL-Null:                       -1470.0
Covariance Type:            nonrobust   LLR p-value:                 3.024e-25
               

In [41]:

print("STEP 5 — RRR TABLE (Model III — Final Model)")
print("Reference: Woman alone (1)")


COMPARISON_LABELS = {
    0: 'Partner alone vs Woman alone',
    1: 'Joint decision vs Woman alone',
    2: 'Other vs Woman alone',
}

rrr_rows = []


if final_model:
    params = final_model.params      # (n_vars × 3)
    conf   = final_model.conf_int()  # MultiIndex (var, cat)
    pvals  = final_model.pvalues     # (n_vars × 3)

    for cat_idx in range(3):
        comp_label = COMPARISON_LABELS[cat_idx]
        cat_str = str(cat_idx + 1)   # matches conf MultiIndex level 0
        
        for var_name in params.index:
            if var_name == 'const':
                continue
            
            # RRR: use params corresponding to this variable and category
            rrr = np.exp(params.loc[var_name][cat_idx])  # params[var_name] over categories
            
            # Confidence interval
            ci = conf.loc[(cat_str, var_name)]
            ci_lo = np.exp(ci[0])
            ci_hi = np.exp(ci[1])
            
            # p-value
            pval = pvals.loc[var_name][cat_idx]
            
            sig = ('***' if pval < 0.001 else
                ('**' if pval < 0.01 else
                ('*'  if pval < 0.05 else 'ns')))
            
            rrr_rows.append({
                'Comparison':   comp_label,
                'Variable':     var_name,
                'RRR':          round(rrr,   3),
                '95% CI Lower': round(ci_lo, 3),
                '95% CI Upper': round(ci_hi, 3),
                'RRR (95% CI)': f"{rrr:.3f} ({ci_lo:.3f}–{ci_hi:.3f})",
                'p-value':      round(pval,  4),
                'Sig.':         sig,
            })
rrr_df = pd.DataFrame(rrr_rows)
print(rrr_df[['Comparison','Variable',
              'RRR (95% CI)','p-value','Sig.']].to_string(index=False))

STEP 5 — RRR TABLE (Model III — Final Model)
Reference: Woman alone (1)
                   Comparison                     Variable          RRR (95% CI)  p-value Sig.
 Partner alone vs Woman alone                age_group_2.0   0.552 (0.243–1.254)   0.1557   ns
 Partner alone vs Woman alone                age_group_3.0   0.357 (0.150–0.846)   0.0193    *
 Partner alone vs Woman alone                age_group_4.0   0.607 (0.244–1.513)   0.2842   ns
 Partner alone vs Woman alone                age_group_5.0   0.343 (0.126–0.935)   0.0365    *
 Partner alone vs Woman alone                age_group_6.0   0.321 (0.104–0.991)   0.0482    *
 Partner alone vs Woman alone                age_group_7.0   0.428 (0.120–1.523)   0.1900   ns
 Partner alone vs Woman alone               marital_status   0.382 (0.250–0.584)   0.0000  ***
 Partner alone vs Woman alone                edu_woman_1.0   1.765 (0.833–3.738)   0.1380   ns
 Partner alone vs Woman alone                edu_woman_2.0   1.062 (0.471

In [43]:

print("\n\nExporting to Excel...")

wb = Workbook()

# ── Style helpers ─────────────────────────────────────────
HFILL  = PatternFill("solid", fgColor="1F4E79")
SFILL  = PatternFill("solid", fgColor="2E75B6")
AFILL  = PatternFill("solid", fgColor="EBF3FB")
LFILL  = PatternFill("solid", fgColor="D6E4F0")
GFILL  = PatternFill("solid", fgColor="C6EFCE")
YFILL  = PatternFill("solid", fgColor="FFF2CC")
RFILL  = PatternFill("solid", fgColor="FFD7D7")
NOFILL = PatternFill()

HFONT  = Font(name="Arial", bold=True,  color="FFFFFF", size=10)
SFONT  = Font(name="Arial", bold=True,  color="FFFFFF", size=10)
BFONT  = Font(name="Arial", size=10)
BBFONT = Font(name="Arial", bold=True,  size=10)
IFONT  = Font(name="Arial", italic=True, size=9,  color="595959")

thin   = Side(style='thin', color='BFBFBF')
BORD   = Border(left=thin, right=thin, top=thin, bottom=thin)
CTR    = Alignment(horizontal='center', vertical='center', wrap_text=True)
LFT    = Alignment(horizontal='left',   vertical='center', wrap_text=True)

def hdr(ws, r, c, text, span=1, sub=False):
    cell = ws.cell(r, c, text)
    cell.fill      = SFILL if sub else HFILL
    cell.font      = SFONT if sub else HFONT
    cell.alignment = CTR
    cell.border    = BORD
    if span > 1:
        ws.merge_cells(
            start_row=r, start_column=c,
            end_row=r,   end_column=c+span-1)
    return cell

def cell(ws, r, c, val, alt=False, bold=False,
         center=False, fill=None, color=None):
    cl = ws.cell(r, c, val)
    cl.fill      = fill if fill else (AFILL if alt else NOFILL)
    cl.font      = Font(name="Arial", bold=bold, size=10,
                        color=color or "000000")
    cl.alignment = CTR if center else LFT
    cl.border    = BORD
    return cl

def aw(ws, mn=8, mx=40):
    for col in ws.columns:
        w = max((len(str(c.value or '')) for c in col), default=mn)
        ws.column_dimensions[
            get_column_letter(col[0].column)].width = min(w+3, mx)

def section_row(ws, r, text, ncols):
    ws.merge_cells(start_row=r, start_column=1,
                   end_row=r,   end_column=ncols)
    cl = ws.cell(r, 1, f"  {text}")
    cl.fill = LFILL
    cl.font = Font(name="Arial", bold=True, size=10, color="1F4E79")
    cl.alignment = LFT

# ────────────────────────────────────────────────────────
# Sheet 0 — Cover
# ────────────────────────────────────────────────────────
ws0 = wb.active
ws0.title = "Cover"
ws0.merge_cells('A1:F2')
ws0['A1'] = ("Women's Family Planning Decisional Autonomy\n"
             "Cameroon DHS 2018 — Regression Analysis")
ws0['A1'].font      = Font(name="Arial", bold=True, size=14, color="1F4E79")
ws0['A1'].alignment = CTR

meta_rows = [
    ("Dataset",       "Cameroon DHS 2018 (CMWR71FL)"),
    ("Sample",        f"N = {len(df):,} women currently in union, non-pregnant"),
    ("Outcome",       "Family planning decisional autonomy (4 categories)"),
    ("Reference",     "Woman decides alone (code 1)"),
    ("Framework",     "UNFPA SDG 5.6.1 — 4-level determinants framework"),
    ("Models",        "Null / Model I (Individual) / Model II (Community) / Model III (Both)"),
    ("Best model",    f"{best_model_name} — selected by lowest AIC"),
    ("Software",      "Python 3 (statsmodels, scikit-learn)"),
]
for i, (k, v) in enumerate(meta_rows, 4):
    ws0.cell(i, 1, k).font  = BBFONT
    ws0.cell(i, 2, v).font  = BFONT
ws0.column_dimensions['A'].width = 18
ws0.column_dimensions['B'].width = 65

# ────────────────────────────────────────────────────────
# Sheet 1 — Table 1: Descriptive Statistics
# ────────────────────────────────────────────────────────
ws1 = wb.create_sheet("Table1_Descriptive")
ws1.merge_cells('A1:D1')
ws1['A1'] = ("Table 1. Sociodemographic and reproductive characteristics "
             "of women currently in union, Cameroon DHS 2018")
ws1['A1'].font      = Font(name="Arial", bold=True, size=12)
ws1['A1'].alignment = LFT

for c, h in enumerate(['Variable', 'Category',
                        'N (weighted)', 'Percent (%)'], 1):
    hdr(ws1, 3, c, h)

row = 4
prev_level = None
for var, label, level in VAR_DEFS:
    if var not in desc_tables:
        continue
    if level != prev_level:
        section_row(ws1, row, level, 4)
        row += 1
        prev_level = level

    ws1.merge_cells(f'A{row}:D{row}')
    cl = ws1.cell(row, 1, f"    {label}")
    cl.font = BBFONT; cl.alignment = LFT
    row += 1

    tab = desc_tables[var]
    for j, (idx, r_) in enumerate(tab.iterrows()):
        alt = j % 2 == 1
        cell(ws1, row, 1, "",           alt=alt)
        cell(ws1, row, 2, str(idx),     alt=alt)
        cell(ws1, row, 3, int(r_['N (weighted)']),
             alt=alt, center=True)
        cell(ws1, row, 4, float(r_['Percent (%)']),
             alt=alt, center=True)
        ws1.cell(row, 4).number_format = '0.0'
        row += 1

# Outcome at bottom
section_row(ws1, row, "OUTCOME VARIABLE", 4);  row += 1
ws1.merge_cells(f'A{row}:D{row}')
cl = ws1.cell(row, 1, "    Family planning decisional autonomy")
cl.font = BBFONT; cl.alignment = LFT; row += 1

for j, (idx, r_) in enumerate(desc_tables['_outcome'].iterrows()):
    alt = j % 2 == 1
    cell(ws1, row, 1, "",    alt=alt)
    cell(ws1, row, 2, str(idx), alt=alt)
    cell(ws1, row, 3, int(r_['N (weighted)']),   alt=alt, center=True)
    cell(ws1, row, 4, float(r_['Percent (%)']), alt=alt, center=True)
    ws1.cell(row, 4).number_format = '0.0'
    row += 1

ws1.cell(row+1, 1,
    "Note: Frequencies are weighted using the DHS sampling weight (V005/1,000,000).").font = IFONT
aw(ws1)

# ────────────────────────────────────────────────────────
# Sheet 2 — Table 2: Bivariate Analysis
# ────────────────────────────────────────────────────────
ws2 = wb.create_sheet("Table2_Bivariate")
ws2.merge_cells('A1:H1')
ws2['A1'] = ("Table 2. Bivariate associations between independent variables "
             "and family planning decisional autonomy (Chi-square analysis)")
ws2['A1'].font      = Font(name="Arial", bold=True, size=12)
ws2['A1'].alignment = LFT

for c, h in enumerate(['Level', 'Variable', 'Chi-square', 'df',
                        'p-value', 'Sig.', "Cramér's V",
                        'Enter model\n(p<0.20)'], 1):
    hdr(ws2, 3, c, h)

prev_level = None
data_row = 4
for j, row_d in bivariate_df.iterrows():
    if row_d['Level'] != prev_level:
        section_row(ws2, data_row, row_d['Level'], 8)
        data_row  += 1
        prev_level = row_d['Level']
    alt = j % 2 == 1
    for c, v in enumerate([row_d['Level'], row_d['Variable'],
                            row_d['Chi-square'], row_d['df'],
                            row_d['p-value'], row_d['Sig.'],
                            row_d["Cramér's V"],
                            row_d['Enter (p<0.20)']], 1):
        cl = cell(ws2, data_row, c, v, alt=alt,
                  center=(c not in [1, 2]))
    # Colour "Enter" cell
    enter_cl = ws2.cell(data_row, 8)
    if row_d['Enter (p<0.20)'] == 'YES':
        enter_cl.fill = GFILL
        enter_cl.font = Font(name="Arial", bold=True,
                             color="375623", size=10)
    data_row += 1

ws2.cell(data_row+1, 1,
    "Note: *** p<0.001  ** p<0.01  * p<0.05  † p<0.20  "
    "ns = not significant. Variables with p<0.20 entered the multivariable model."
).font = IFONT
aw(ws2)


# ────────────────────────────────────────────────────────
# Sheet 3 — Table 3: Model Comparison
# ────────────────────────────────────────────────────────
ws4 = wb.create_sheet("Table4_ModelComparison")
ws4.merge_cells('A1:F1')
ws4['A1'] = ("Table 3. Multinomial logistic regression — Model comparison "
             "(reference: Woman alone)")
ws4['A1'].font      = Font(name="Arial", bold=True, size=12)
ws4['A1'].alignment = LFT

metrics_list = list(comparison_df.columns)
model_names  = list(comparison_df.index)

# Header
hdr(ws4, 3, 1, 'Metric')
for c, mn in enumerate(model_names, 2):
    hdr(ws4, 3, c, mn)

for r, metric in enumerate(metrics_list, 4):
    alt = r % 2 == 1
    cell(ws4, r, 1, metric, alt=alt, bold=True)
    for c, mn in enumerate(model_names, 2):
        val = comparison_df.loc[mn, metric]
        cl  = cell(ws4, r, c, val, alt=alt, center=True)
        # Highlight best (lowest AIC / lowest Deviance / highest R²)
        if metric == 'AIC' and mn == best_model_name:
            cl.fill = GFILL
        if metric == 'Deviance' and mn == best_model_name:
            cl.fill = GFILL

# Note row
note_r = len(metrics_list) + 6
ws4.cell(note_r, 1,
    f"Best model: {best_model_name} (lowest AIC). "
    "Model III includes both individual- and community-level variables. "
    "McFadden pseudo-R²: values >0.20 indicate excellent fit."
).font = IFONT
aw(ws4)

# ────────────────────────────────────────────────────────
# Sheet 5 — Table 5: RRR Table (final model)
# ────────────────────────────────────────────────────────
ws5 = wb.create_sheet("Table5_RRR")
ws5.merge_cells('A1:H1')
ws5['A1'] = ("Table 5. Multinomial logistic regression — Relative Risk Ratios "
             f"({best_model_name}; reference: Woman alone)")
ws5['A1'].font      = Font(name="Arial", bold=True, size=12)
ws5['A1'].alignment = LFT
ws5['A2'] = ("Results expressed as Relative Risk Ratios (RRR) with 95% "
             "confidence intervals. Reference category = Woman alone (1).")
ws5['A2'].font      = IFONT
ws5['A2'].alignment = LFT

for c, h in enumerate(['Comparison', 'Variable', 'RRR',
                        '95% CI Lower', '95% CI Upper',
                        'RRR (95% CI)', 'p-value', 'Sig.'], 1):
    hdr(ws5, 4, c, h)

prev_comp = None
data_row  = 5
for j, row_d in rrr_df.iterrows():
    comp = row_d['Comparison']
    if comp != prev_comp:
        # Comparison section header
        ws5.merge_cells(
            start_row=data_row, start_column=1,
            end_row=data_row,   end_column=8)
        cl = ws5.cell(data_row, 1, f"  {comp}")
        cl.fill = SFILL; cl.font = SFONT; cl.alignment = LFT
        data_row  += 1
        prev_comp  = comp

    alt = j % 2 == 1
    sig = row_d['Sig.']
    rrr_val = row_d['RRR']

    for c, v in enumerate([comp, row_d['Variable'],
                            row_d['RRR'],
                            row_d['95% CI Lower'],
                            row_d['95% CI Upper'],
                            row_d['RRR (95% CI)'],
                            row_d['p-value'],
                            sig], 1):
        cl = cell(ws5, data_row, c, v, alt=alt,
                  center=(c not in [1, 2, 6]))

    # Colour-code significance
    sig_cl = ws5.cell(data_row, 8)
    if sig in ['***', '**', '*']:
        sig_cl.fill = YFILL
        sig_cl.font = Font(name="Arial", bold=True,
                           color="7D6608", size=10)
    # Colour RRR: green if <1 (protective), red if >1 (risk)
    rrr_cl = ws5.cell(data_row, 3)
    if rrr_val < 1 and sig in ['***', '**', '*']:
        rrr_cl.font = Font(name="Arial", bold=True,
                           color="1E8449", size=10)
    elif rrr_val > 1 and sig in ['***', '**', '*']:
        rrr_cl.font = Font(name="Arial", bold=True,
                           color="922B21", size=10)
    data_row += 1

ws5.cell(data_row+1, 1,
    "Note: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant. "
    "RRR = Relative Risk Ratio. Green RRR = reduces relative risk vs reference. "
    "Red RRR = increases relative risk vs reference. Survey weights applied."
).font = IFONT
aw(ws5)

# ── Save ──────────────────────────────────────────────────
out = "final_regression_results.xlsx"
wb.save(out)
print(f"\n✅ Saved: {out}")
print("\nSheets:")
for s in wb.sheetnames:
    print(f"  - {s}")



Exporting to Excel...

✅ Saved: final_regression_results.xlsx

Sheets:
  - Cover
  - Table1_Descriptive
  - Table2_Bivariate
  - Table4_ModelComparison
  - Table5_RRR
